# Parse an ASPECT statistics file

**Author:** Haoyuan Li

ASPECT writes quantities collected by its postprocessors to a `statistics` file. The postprocessors also name the columns in the file header, so `AspectStatistics` reads the column descriptions from the header of the `statistics` file instead of assuming a fixed layout.

In [ ]:
from pathlib import Path

from gdmate.aspect_tools import AspectStatistics

# Track the repository root
repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
)

## Load a statistics table

This example fixture comes from a real ASPECT model. To analyze another file, replace `statistics_path` with the path to the new `output/statistics` file. The parsed object will contain a `data` attribute that stores a pandas DataFrame with a shape of (rows, columns).

In [ ]:
statistics_path = (
    repository_root
    / "tests"
    / "data"
    / "aspect_tools"
    / "statistics_excerpt.txt"
)
statistics = AspectStatistics(statistics_path)
statistics.data.shape

## Inspect and select columns

As the underlying `data` object is a pandas `DataFrame`, so standard pandas selection, filtering, and plotting methods are available. Units in trailing parentheses are separated from column names: for example, `Time (years)` becomes the `Time` column, and its unit is available as `statistics.units["Time"]`. A column without a stated unit maps to `None`.

In [ ]:
for number, name in enumerate(statistics.data.columns, start=1):
    print(f"{number:2}: {name} [{statistics.units[name]}]")

statistics.data.head()

In [ ]:
steps = statistics.data["Time step number"]
times = statistics.data["Time"]
timestep_sizes = statistics.data["Time step size"]

list(zip(steps[:5], times[:5], timestep_sizes[:5], strict=True))

## Match visualization filenames with timestep statistics (not recommended)

When present, `Visualization file name` contains the filenames ASPECT recorded in the statistics table. Empty strings indicate rows without a recorded filename. Filter this column to compare the recorded filenames with the timestep and time values in the same rows.

To list graphical output events, use `AspectLogParser.parse_graphical_outputs()` on the ASPECT log. The log can include writes during initial refinement that have no associated timestep, so its event list can differ from the filenames recorded in the statistics table.

In [ ]:
has_solution = statistics.data["Visualization file name"] != ""
statistics.data.loc[
    has_solution, ["Time step number", "Time", "Visualization file name"]
]